In [2]:
import pandas as pd

In [3]:
books = pd.read_csv("books_with_categories")

In [4]:
from transformers import pipeline
classifier = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    top_k=None, 
    device="mps"
)
classifier("I love this!")


Device set to use mps


[[{'label': 'surprise', 'score': 0.48696979880332947},
  {'label': 'neutral', 'score': 0.2234414517879486},
  {'label': 'joy', 'score': 0.14913064241409302},
  {'label': 'anger', 'score': 0.07174026966094971},
  {'label': 'sadness', 'score': 0.04664652422070503},
  {'label': 'disgust', 'score': 0.01629774644970894},
  {'label': 'fear', 'score': 0.005773560609668493}]]

In [5]:
description = books["description"][0]
classifier(description)

[[{'label': 'fear', 'score': 0.654841423034668},
  {'label': 'neutral', 'score': 0.16985200345516205},
  {'label': 'sadness', 'score': 0.11640875786542892},
  {'label': 'surprise', 'score': 0.02070065401494503},
  {'label': 'disgust', 'score': 0.019100766628980637},
  {'label': 'joy', 'score': 0.015161258168518543},
  {'label': 'anger', 'score': 0.003935154993087053}]]

In [6]:
pred = classifier(description.split("."))
pred

[[{'label': 'surprise', 'score': 0.7296027541160583},
  {'label': 'neutral', 'score': 0.1403856724500656},
  {'label': 'fear', 'score': 0.06816212832927704},
  {'label': 'joy', 'score': 0.047942448407411575},
  {'label': 'anger', 'score': 0.009156345389783382},
  {'label': 'disgust', 'score': 0.0026284728664904833},
  {'label': 'sadness', 'score': 0.002122161677107215}],
 [{'label': 'neutral', 'score': 0.44937166571617126},
  {'label': 'disgust', 'score': 0.2735905349254608},
  {'label': 'joy', 'score': 0.10908280313014984},
  {'label': 'sadness', 'score': 0.09362749755382538},
  {'label': 'anger', 'score': 0.040478236973285675},
  {'label': 'surprise', 'score': 0.02697022631764412},
  {'label': 'fear', 'score': 0.006879068911075592}],
 [{'label': 'neutral', 'score': 0.6462157368659973},
  {'label': 'sadness', 'score': 0.2427336573600769},
  {'label': 'disgust', 'score': 0.04342268407344818},
  {'label': 'surprise', 'score': 0.028300486505031586},
  {'label': 'joy', 'score': 0.01421142

In [10]:
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

def calculate_max_emotion_scores(predictions):
    per_emotion_score = {label: [] for label in emotion_labels}
    for prediction in predictions:
        for pred in prediction:
            per_emotion_score[pred["label"]].append(pred["score"])

    return {label: max(score) for label, score in per_emotion_score.items()}




In [11]:
from tqdm import tqdm
isbn = []
emotion_score = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    description = books["description"][i].split(".")
    predictions = classifier(description)
    max_scores = calculate_max_emotion_scores(predictions)
    for label, scores in max_scores.items():
        emotion_score[label].append(scores)


100%|██████████| 5197/5197 [03:56<00:00, 21.94it/s]


In [12]:
emotions_df = pd.DataFrame(emotion_score)
emotions_df["isbn13"] = isbn

In [13]:
emotions_df

,anger,disgust,fear,joy,neutral,sadness,surprise,isbn13
0,0.051973,0.273591,0.928169,0.932798,0.646216,0.967157,0.729603,9780002005883
1,0.612619,0.348285,0.942528,0.704421,0.887940,0.074825,0.252545,9780002261982
2,0.051973,0.157667,0.972321,0.767237,0.608933,0.074825,0.046931,9780006178736
3,0.351483,0.157667,0.360707,0.251881,0.732687,0.074825,0.046931,9780006280897
4,0.081412,0.184495,0.095043,0.035207,0.925904,0.475881,0.046931,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.148209,0.030642,0.919165,0.255170,0.853722,0.980877,0.030656,9788172235222
5193,0.057145,0.157667,0.038787,0.400263,0.883199,0.074825,0.227765,9788173031014
5194,0.009997,0.009929,0.339217,0.947779,0.375756,0.066685,0.057625,9788179921623
5195,0.051973,0.157667,0.459269,0.759455,0.951104,0.368110,0.067808,9788185300535


In [14]:
books = pd.merge(books, emotions_df, on="isbn13", how="left")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,neutral,sadness,surprise
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.051973,0.273591,0.928169,0.932798,0.646216,0.967157,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612619,0.348285,0.942528,0.704421,0.887940,0.074825,0.252545
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.051973,0.157667,0.972321,0.767237,0.608933,0.074825,0.046931
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.157667,0.360707,0.251881,0.732687,0.074825,0.046931
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.035207,0.925904,0.475881,0.046931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.148209,0.030642,0.919165,0.255170,0.853722,0.980877,0.030656
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.057145,0.157667,0.038787,0.400263,0.883199,0.074825,0.227765
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339217,0.947779,0.375756,0.066685,0.057625
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.051973,0.157667,0.459269,0.759455,0.951104,0.368110,0.067808


In [15]:
books.describe()

,isbn13,published_year,average_rating,num_pages,ratings_count,anger,disgust,fear,joy,neutral,sadness,surprise
count,5.197000e+03,5197.000000,5197.000000,5197.000000,5.197000e+03,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000
mean,9.780667e+12,1999.804118,3.922246,348.472195,2.113112e+04,0.156067,0.231809,0.297079,0.274506,0.757189,0.208756,0.170970
std,5.951053e+08,9.082979,0.324975,229.891672,1.446480e+05,0.219279,0.197785,0.343570,0.320091,0.188142,0.253915,0.204677
min,9.780002e+12,1876.000000,0.000000,0.000000,0.000000e+00,0.000606,0.000821,0.000442,0.000550,0.000981,0.001251,0.000779
25%,9.780313e+12,1998.000000,3.750000,213.000000,1.830000e+02,0.051973,0.157667,0.038787,0.020884,0.608933,0.074825,0.046931
50%,9.780521e+12,2002.000000,3.940000,312.000000,1.125000e+03,0.051973,0.157667,0.086226,0.094542,0.808767,0.074825,0.065274
75%,9.780807e+12,2005.000000,4.120000,416.000000,6.574000e+03,0.132362,0.185342,0.544719,0.497997,0.925201,0.196813,0.205456
max,9.789028e+12,2019.000000,5.000000,3342.000000,5.629932e+06,0.989582,0.989417,0.995326,0.992068,0.974344,0.989361,0.983455


In [16]:
books.to_csv("books_with_emotions.csv", index=False)